<a href="https://colab.research.google.com/github/Kanchanajaddu/GEN_AI-exercises/blob/main/knowledgeass(w1d5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
#given question:
#Build a Knowledge Assistant (like a mini ChatGPT) that:
#Lets user upload a set of documents (PDF/TXT/Markdown).
#Converts them into embeddings using OpenAI Embeddings API.
#Stores embeddings in a vector DB (FAISS for local or Pinecone/Chroma for cloud).
#When the user asks a question, it: Retrieves the most relevant chunks from the DB.
#Feeds them into OpenAI ChatCompletion API with a carefully designed prompt.
#Returns a final answer in natural language.

In [ ]:
#open ai api key
import os
openai_api_key = input("Please enter your OpenAI API key (e.g., sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx): ")
os.environ["OPENAI_API_KEY"] = openai_api_key
print("OpenAI API key configured successfully.")

In [3]:
pip install langchain openai tiktoken pypdf faiss-cpu

In [4]:
pip install langchain-community langchain-openai

In [5]:
#upload documents
from google.colab import files
import os
def upload_files():
    uploaded = files.upload()
    file_paths = []
    for filename in uploaded.keys():
        print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')
        file_paths.append(filename)
    return file_paths
corpus_files = upload_files()


Saving health care policy document 3.pdf to health care policy document 3 (1).pdf
User uploaded file "health care policy document 3 (1).pdf" with length 438003 bytes


In [8]:
#loading and processing the documents
from langchain_community.document_loaders import PyPDFLoader, TextLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
def load_documents(file_paths):
    docs = []
    for file_path in file_paths:
        _, file_extension = os.path.splitext(file_path)
        if file_extension == '.pdf':
            loader = PyPDFLoader(file_path)
        elif file_extension == '.txt':
            loader = TextLoader(file_path)
        elif file_extension == '.md':
            loader = TextLoader(file_path)
        else:
            print(f"Skipping unsupported file type: {file_path}")
            continue
        docs.extend(loader.load())
    return docs

def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap  = 200,
        length_function = len,
    )
    return text_splitter.split_documents(documents)

# Load and split documents
raw_documents = load_documents(corpus_files)
text_chunks = split_documents(raw_documents)

print(f"Loaded {len(raw_documents)} raw documents and split into {len(text_chunks)} text chunks.")

Loaded 27 raw documents and split into 125 text chunks.


In [11]:
#create embeddings and vector store
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(text_chunks, embeddings)

print("Vector store created successfully using FAISS.")

Vector store created successfully using FAISS.


In [14]:
#set up the knowledge assistant
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Create the RAG chain using LCEL
qa_chain = (
    { "context": itemgetter("question") | vectorstore.as_retriever(), "question": itemgetter("question") }
    | prompt
    | llm
)

print("Knowledge Assistant (RAG chain) initialized. You can now ask questions!")

Knowledge Assistant (RAG chain) initialized. You can now ask questions!


In [19]:
#asking questions
from IPython.display import display, Markdown

def ask_question(question):
    if question:
        print(f"\nQuestion: {question}")
        result = qa_chain.invoke({"question": question})
        answer = result.content
        print(f"\nAnswer: {answer}")
    else:
        print("Please enter a question.")
while True:
    user_question = input("Your question (type 'exit' to quit): ")
    if user_question.lower() == 'exit':
        print("Exiting Q&A session.")
        break
    ask_question(user_question)

Your question (type 'exit' to quit): what is out patient department

Question: what is out patient department

Answer: An Out Patient Department is a department where a patient is not hospitalized and is being treated in an office, clinic, or other ambulatory care facility by a Medical Practitioner for illness/disease.
Your question (type 'exit' to quit): exit
Exiting Q&A session.
